In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("CNN development started")

CNN development started


building the model

In [2]:

class NeuroVisionCNN(nn.Module):

    def __init__(self, num_classes=4):
        super(NeuroVisionCNN, self).__init__()

        # Block 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)

        # Block 2
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)

        # Block 3
        self.conv3 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)

        # Block 4
        self.conv4 = nn.Conv2d(128, 256, kernel_size=3, padding=1)
        self.bn4 = nn.BatchNorm2d(256)

        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Reduce feature map dimensions
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        # Classification layers
        self.dropout = nn.Dropout(0.4)
        self.fc1 = nn.Linear(256, 128)
        self.fc2 = nn.Linear(128, num_classes)

    def forward(self, x):

        # Block 1
        x = self.pool(F.relu(self.bn1(self.conv1(x))))

        # Block 2
        x = self.pool(F.relu(self.bn2(self.conv2(x))))

        # Block 3
        x = self.pool(F.relu(self.bn3(self.conv3(x))))

        # Block 4
        x = self.pool(F.relu(self.bn4(self.conv4(x))))

        # Global Average Pooling
        x = self.global_pool(x)

        # Flatten
        x = torch.flatten(x, 1)

        # Classification
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)

        return x


In [3]:
device = torch.device("cpu")

model = NeuroVisionCNN(num_classes=4)
model = model.to(device)

print(model)

NeuroVisionCNN(
  (conv1): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv3): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn3): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (conv4): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (global_pool): AdaptiveAvgPool2d(output_size=(1, 1))
  (dropout): Dropout(p=0.4, inplace=False)
  (fc1): Linear(in_features=256, out_features=128, bias=True)
  (fc2): Linear(in_features

In [4]:
total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Total parameters:", f"{total_params:,}")
print("Trainable parameters:", f"{trainable_params:,}")
print("Model device:", next(model.parameters()).device)

Total parameters: 422,788
Trainable parameters: 422,788
Model device: cpu


In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import create_data_loaders

train_loader, val_loader, test_loader = create_data_loaders()

print("Training batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Testing batches:", len(test_loader))

========== NEUROVISION DATASET ==========
Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Class mapping: {'glioma': 0, 'meningioma': 1, 'notumor': 2, 'pituitary': 3}

Training images: 4480
Validation images: 1120
Testing images: 1600

Training batches: 280
Validation batches: 70
Testing batches: 100

Image size: 224
Batch size: 16

Data loading completed successfully!
Training batches: 280
Validation batches: 70
Testing batches: 100


testing the model with one batch

In [6]:
images, labels = next(iter(train_loader))

images = images.to(device)

model.eval()

with torch.no_grad():
    outputs = model(images)

print("Input shape:", images.shape)
print("Output shape:", outputs.shape)
print("Expected output shape:", (16, 4))

Input shape: torch.Size([16, 3, 224, 224])
Output shape: torch.Size([16, 4])
Expected output shape: (16, 4)


checking the classiication clsss


In [10]:

class_names = train_loader.dataset.dataset.classes

print("Class mapping:")

for index, class_name in enumerate(class_names):
    print(index, ":", class_name)

Class mapping:
0 : glioma
1 : meningioma
2 : notumor
3 : pituitary


importing the training library

In [11]:
import torch
import torch.nn as nn
import torch.optim as optim

import matplotlib.pyplot as plt
import time
from pathlib import Path

loss fn and optimizer

In [12]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)
print("Initial learning rate:", optimizer.param_groups[0]["lr"])

Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)
Initial learning rate: 0.001


In [13]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:

        images = images.to(device)
        labels = labels.to(device)

        # Clear previous gradients
        optimizer.zero_grad()

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Statistics
        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy

the validation fn

In [15]:
def validate_model(model, loader, criterion, device):

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    epoch_loss = running_loss / total
    epoch_accuracy = 100 * correct / total

    return epoch_loss, epoch_accuracy

In [16]:
EPOCHS = 10

history = {
    "train_loss": [],
    "val_loss": [],
    "train_accuracy": [],
    "val_accuracy": []
}

best_val_loss = float("inf")

# Create model directory
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

BEST_MODEL_PATH = MODEL_DIR / "neurovision_best.pth"

start_time = time.time()

for epoch in range(EPOCHS):

    epoch_start = time.time()

    # Training
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer,
        device
    )

    # Validation
    val_loss, val_acc = validate_model(
        model,
        val_loader,
        criterion,
        device
    )

    # Update learning rate
    scheduler.step(val_loss)

    # Store history
    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)

    history["train_accuracy"].append(train_acc)
    history["val_accuracy"].append(val_acc)

    # Save best model
    if val_loss < best_val_loss:

        best_val_loss = val_loss

        torch.save({
            "epoch": epoch + 1,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "val_loss": val_loss,
            "class_names": class_names
        }, BEST_MODEL_PATH)

        status = "Best model saved"

    else:
        status = ""

    epoch_time = time.time() - epoch_start

    print(f"\nEpoch [{epoch+1}/{EPOCHS}]")
    print(f"Training Loss: {train_loss:.4f}")
    print(f"Training Accuracy: {train_acc:.2f}%")
    print(f"Validation Loss: {val_loss:.4f}")
    print(f"Validation Accuracy: {val_acc:.2f}%")
    print(f"Time: {epoch_time:.2f} seconds")
    print(status)

total_time = time.time() - start_time

print("\nTraining completed!")
print(f"Total training time: {total_time / 60:.2f} minutes")
print("Best model saved at:", BEST_MODEL_PATH)


Epoch [1/10]
Training Loss: 0.8245
Training Accuracy: 67.28%
Validation Loss: 0.6131
Validation Accuracy: 79.02%
Time: 513.63 seconds
Best model saved

Epoch [2/10]
Training Loss: 0.6436
Training Accuracy: 75.94%
Validation Loss: 0.5734
Validation Accuracy: 78.04%
Time: 523.91 seconds
Best model saved


KeyboardInterrupt: 

Visualize training performance

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss graph
axes[0].plot(
    history["train_loss"],
    label="Training Loss"
)

axes[0].plot(
    history["val_loss"],
    label="Validation Loss"
)

axes[0].set_title("Training vs Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()
axes[0].grid(True)

# Accuracy graph
axes[1].plot(
    history["train_accuracy"],
    label="Training Accuracy"
)

axes[1].plot(
    history["val_accuracy"],
    label="Validation Accuracy"
)

axes[1].set_title("Training vs Validation Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy (%)")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

Loading  the best saved model

In [ ]:
checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=device,
    weights_only=True
)

model.load_state_dict(checkpoint["model_state_dict"])

model.eval()

print("Best epoch:", checkpoint["epoch"])
print("Best validation loss:", checkpoint["val_loss"])
print("Best model loaded successfully!")